# Welcome to ToxiScan!

## About this Notebook
This interactive Jupyter Notebook presents **ToxiScan**, a Python package designed to evaluate  the toxicity of molecules. By simply entering a molecule, Toxiscan retrieves its structure from the PubChem database, detects toxic functional groups, the **toxicophores**, computes a toxicity score, and visualizes the molecule in 3D highlighting the toxicophores. 

This package was created as a collaborative project for the EPFL course Practical Programming in Chemistry.

## Why does molecular toxicity matter? 
In drug discovery and chemical safety, identifying toxic functional groups early is crucial. **Toxicophores** (nitro groups, epoxides or aldehydes) are specific chemical substructures known to cause adverse biological effects. ToxiScan automates their detection to help chemists quickly flag potentially harmful molecules. 
Avoiding the synthesis of toxic molecules is also a **green chemistry** approach, it saves resources and reduces dangerous chemical waste. 

## Why choose ToxiScan? 
Several tools already exist for toxicity prediction: 
- **SwissADME** : a free online tool, very complete, detects PAINS and structural alerts
- **pkCSM** : predicts toxicity using machine learning
- **FAF-Drugs** : filters toxic molecules 
- **RDKit itself** has integrated toxicity filters

But the real added value of ToxiScan is **accessibility and integration**:
- SwissADME is a web interface not integrable in an automated Python script 
- ToxiScan is a **pip-installable Python package** usable in ay cheminformatics workflow
- Ideal for analyzing **entire lists of molecules** automatically, not just one at a time 
- **Open source and transparent**, you can see exactly which rules are applied, not a black box

*ToxiScan does not reinvent the wheel, but makes toxicity analysis **accessible in Python**, integrable in any cheminformatics pipeline.*


## Let's get started
Please run the following cell to import all the fucntions from ToxiScan and ensure the code runs properly.  

In [ ]:
from toxiscan import (get_smiles, get_molecule_info, find_toxicophores,
                      count_toxicophores, remove_redundant_toxicophores,
                      toxicity_approximation, compute_properties,
                      interpret_properties, properties_toxicity,
                      draw_molecule, draw_molecule_3d)

## 1. Retrieve a moleculefrom PubChem
The function 'get_molecule_info(molecule_name)' takes a molecule name as input and retrieves its basic chemical information from the **PubChem** database using the PubChem REST API. 

**What is a SMILES?**
SMILES (Simplified Molecular Input Line Entry System) is a notation that encodes the structure of a molecule as a string of characters. 
For example, aspirin is encoded as 'CC(=O)Oc1ccccc1C(=O)O'

**Input:**
- 'molecule_name' (str): the common name of the molecule (e.g. "aspirin", "caffein")
**Output:**
- A dictionnary containing the name, molecular formula, molecular weight, IUPAC name and SMILES for the molecule. 

**How does it works?**
The function sends a GET request to the PubChem REST API using the molecule name, then parses the JSON response to extract the relevant chemical properties. 

**Limitations:**
The function only accepts the common name of the molecule. If the name is slightly different from what PubChem expects, the molecule will not be found.

### Example output
For aspririn, you should get: 
Name: aspirin
Formula: C9H8O4
Molecular weight: 180.2 g/mol
IUPAC name: 2-acetyloxybenzoic acid
SMILES: CC(=O)Oc1ccccc1C(=O)O

**Let's try it! Run the cell below:**

In [ ]:
molecule_name = "aspirin"
info = get_molecule_info(molecule_name)
print(f"Name: {info['name']}")
print(f"Formula: {info['formula']}")
print(f"Molecular weight: {info['molecular_weight']} g/mol")
print(f"IUPAC name: {info['iupac_name']}")
print(f"SMILES: {info['smiles']}") 

## 2. Detect toxicophores
The function 'find_toxicophores(smiles)' scans the molecule for toxic functional groups using **SMARTS pattern matching** from RDKit.
ToxiScan recognizes **26 toxicophores** organized in 8 categories: 
| Category | Examples |
|----------|----------|
| Nitrogen-based | Nitro group, Nitroso group, Primary aniline, Hydrazine, Aromatic azo, N-oxide, Hydroxamic acid |
| Carbonyl-based | Aldehyde, Acyl halide, Anhydride, Alpha halo ketone, Beta lactone |
| Electrophilic | Epoxide, Isocyanate, Michael acceptor, Aziridine, Activated alkyne |
| Halogen-based | Alkyl halide, Allylic halide |
| Sulfur-based | Thiol, Thiocarbonyl |
| Peroxide | Peroxide |
| Polycyclic aromatic | Quinone, Coumarin | 
| Reactie oxygen | Hydroperoxide |

**Input:**
- 'smiles' (str): the SMILES string of the molecule 

**Output:**
- A dictionnary where keys are toxicophore names and values are lists of atom indices where the toxicophore was found. 
- Returns an empty dict if no toxicophores are detected. 

**How does it works?**
For each of the 26 toxicophores, the function converts the SMARTS pattern to an RDKit mol object and searches for substructures matches in the molecule. Matched atom indices are collected and returned. 

**Limitations**
SMARTS-based detection can sometimes produce false positives when functional groups overlap. A dedicated cleaning function 'remove_redundant_toxicophores' handles this by removing subgroups that are already captured by a larger detected group. 

### Example output
For aspirin, you should get: 
Toxicophores detected in aspirin 
- Aldehyde (atoms:[7,8])

**Let's try it! Run the cell below:**

In [ ]:
smiles = info["smiles"]
toxicophores_found = find_toxicophores(smiles)

if toxicophores_found:
    print(f"Toxicophores detected in {molecule_name}:")
    for name, atoms in toxicophores_found.items():
        print(f"  - {name} (atoms: {atoms})")
else:
    print(f"No toxicophores detected in {molecule_name}.")

## 3. Visualize the molecule in 2D

The function 'draw-molecule(smiles, toxicophores_found)' draws the molecule using RDKit and highlights all detected toxic atoms in **yellow-green**.

**Input:**
- 'smiles' (str): the SMILES string of the molecule
- 'toxicophores_found' (dict); the dictionary returned by 'find_toxicophores()'

**Output:**
- Displays the molecule as an SVG image inline in the notebook with toxic fragments highlighted in yellow-green

**How does it work?**
The function collects all atom indices from the toxicophores dictionary, the uses RDKit's 'MolDraw2DSVG' with a custom yellow-green highlight color (RGB: 0.6, 0.9, 0.2) to render the molecule. 

### Expected output
You should see the molecule drawn with the toxic atoms highlighted in yellow-green. 

**Let's try it! Run the cell below:**

In [ ]:
from IPython.display import SVG
draw_molecule(smiles, toxicophores_found)
with open("molecule.svg") as f:
    display(SVG(f.read())) 

## 4. 3D Visualization

The function 'draw_molecule_3d(smiles, toxicophores_found)' generates an **interactive 3D visualization** of the molecule using py3Dmol, with toxic atoms highlighted in green. 

**Input:**
- 'smiles' (str): the SMILES string of the molecule 
- 'toxicophores_found' (dict): dictionnary returned by 'find_toxicophores()'

**Output:**
- An interactive 3D viewer directly in the notebook where you can rotate, zoom in and out on the molecule. 

**How does it work?**
The function generates 3D coordinates using RDKit's 'EmbedMolecule', then passes the mol block to py3Dmol for interactive rendering. All atoms are shown as ball-and-stick, and toxic atoms are highlighted in green. 

### Expected output 
You should see an interactive 3D representation of the moelcule with the toxic atoms highlighted in green. ou can rotate and zoom using your mouse. 

**Let's try it! Run the cell below:**

In [ ]:
from IPython.display import HTML
html_3d = draw_molecule_3d(smiles, toxicophores_found)
HTML(html_3d)

## 5. Compute the toxicity score 

The function 'toxicity_approximation(smiles)' computes a normalized toxicity score based on the detected toxicophores. 

**Input:**
- 'smiles' (str): the SMILES string of the molecule 

**Output:**
- A float representing the toxicity score of the molecule. 

**How does it work?**
Each toxicophore is assigned a **weight** based on its known electrophilic reactivity: 

| Weight | Toxicophores |
|---------|-------------|
| 1 (mild) | Thiol, Alkyl halide, N-oxide, Coumarin |
| 2 (moderate) | Nitro group, ALdehyde, Michael acceptor, Aromatic azo |
| 3 (high) | Epoxide, Acyl halide, Peroxide, Isocyanate, Aziridine |

The total weighted score is then divided by the number of atoms in the molecule, so that larger molecules are not unfairly penalized compared to smaller ones. 

**Limitations**
This score is a structural approximation only. It does not account for pharmacokinetics, metabolism, or the biological context of the molecule. Two molecules with the same score can have very differetn actual toxicity in vivo. 

### Example output
For aspirin,
Toxicity score: 0.0526
Number of toxicophores detected: 1

**Let's try it! Run the cell below:**

In [ ]:
score = toxicity_approximation(smiles)
n_tox = count_toxicophores(smiles)
print(f"Toxicity score of {molecule_name}: {score:.4f}")
print(f"Number of toxicophores detected: {n_tox}")